<a href="https://colab.research.google.com/github/shin-noda/leetcode-problemset-97/blob/main/Problem1923.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [29]:
from collections import defaultdict, deque

class SAIS:
    def is_lms(self, i, t):
        return i > 0 and t[i] and not t[i - 1]


    def get_buckets(self, alphabet_size, buckets):
        head = [0] * alphabet_size
        tail = [0] * alphabet_size
        acc = 0

        for i, c in enumerate(buckets):
            head[i] = acc
            acc += c
            tail[i] = acc

        return head, tail


    def induce(self, lms, A, t, alphabet_size, buckets):
        n = len(A)
        sa = [-1] * n

        head, tail = self.get_buckets(alphabet_size, buckets)
        for idx in reversed(lms):
            c = A[idx]
            tail[c] -= 1
            sa[tail[c]] = idx

        head, tail = self.get_buckets(alphabet_size, buckets)
        for i in range(n):
            if sa[i] > 0 and not t[sa[i] - 1]:
                c = A[sa[i] - 1]
                sa[head[c]] = sa[i] - 1
                head[c] += 1

        head, tail = self.get_buckets(alphabet_size, buckets)
        for i in range(n - 1, -1, -1):
            if sa[i] > 0 and t[sa[i] - 1]:
                c = A[sa[i] - 1]
                tail[c] -= 1
                sa[tail[c]] = sa[i] - 1

        return sa


    def lms_equal(self, A, t, a, b):
        i = 0

        while True:
            a_end = self.is_lms(a + i, t)
            b_end = self.is_lms(b + i, t)

            if A[a + i] != A[b + i] or t[a + i] != t[b + i]:
                return False

            if i > 0 and a_end and b_end:
                return True

            if a_end != b_end:
                return False

            i += 1


    def process(self, A, alphabet_size):
        n = len(A)

        if n <= 1:
            return list(range(n))

        t = [False] * n
        t[-1] = True
        for i in range(n - 2, -1, -1):
            if A[i] < A[i + 1]:
                t[i] = True

            elif A[i] > A[i + 1]:
                t[i] = False

            else:
                t[i] = t[i + 1]

        buckets = [0] * alphabet_size
        for x in A:
            buckets[x] += 1

        lms = [i for i in range(n) if self.is_lms(i, t)]
        sa = self.induce(lms, A, t, alphabet_size, buckets)
        sorted_lms = [x for x in sa if self.is_lms(x, t)]
        lms_id = [-1] * n
        name = 0
        lms_id[sorted_lms[0]] = 0

        for i in range(1, len(sorted_lms)):
            u = sorted_lms[i - 1]
            v = sorted_lms[i]

            if not self.lms_equal(A, t, u, v):
                name += 1

            lms_id[v] = name

        summary = [lms_id[i] for i in lms]

        if name + 1 < len(summary):
            summary_sa = self.process(summary, name + 1)

        else:
            summary_sa = list(range(len(summary)))

        ordered_lms = [lms[i] for i in summary_sa]

        return self.induce(ordered_lms, A, t, alphabet_size, buckets)

class Solution:
    def longestCommonSubpath(self, n, paths):
        m = len(paths)

        # Concateneate all paths with unique separators
        A = []

        # Tracks which friend owns the element at this index
        owner = []

        # We use negative unique markers, starting from -1 down to -m
        separator = -1
        for friend_id, path in enumerate(paths):
            for city in path:
                A.append(city)
                owner.append(friend_id)

            if friend_id < m - 1:
                A.append(separator)

                # Separators belong to no one
                owner.append(-1)
                separator -= 1

        # SA-IS requires non-negative integers.
        # Shift everything up so min element is 1.
        min_val = min(A)
        A = [x - min_val + 1 for x in A]

        # The virtual terminator for SA-IS
        A.append(0)

        # Run SA-IS
        sais = SAIS()
        suffix_array = sais.process(A, max(A) + 1)

        # Drop the virtual terminator
        suffix_array = suffix_array[1:]

        # Run Kasai's Algorithm to get LCP
        total_len = len(suffix_array)
        lcp = [0] * total_len
        rank = [0] * total_len

        for idx, position in enumerate(suffix_array):
            rank[position] = idx

        h = 0
        for i in range(total_len):
            if rank[i] > 0:
                j = suffix_array[rank[i] - 1]

                # Make sure we don't match the separators (shifted values of negative numbers)
                while (
                    i + h < total_len and j + h < total_len and
                    A[i + h] == A[j + h]
                ):
                       h += 1

            lcp[rank[i]] = h

            if h > 0:
                h -= 1

        left = 0
        friend_counts = defaultdict(int)
        distinct_friends = 0
        max_common_subpath = 0

        # Monotonic queue to efficiently maintain the minimum LCP in our sliding window
        min_lcp_deque = deque()

        for right in range(total_len):
            curr_friend = owner[suffix_array[right]]
            if curr_friend != -1:
                if friend_counts[curr_friend] == 0:
                    distinct_friends += 1
                friend_counts[curr_friend] += 1

            if right > 0:
                while min_lcp_deque and lcp[min_lcp_deque[-1]] >= lcp[right]:
                    min_lcp_deque.pop()
                min_lcp_deque.append(right)

            # Clean out out-of-bounds indices from the front of our deque
            while min_lcp_deque and min_lcp_deque[0] <= left:
                min_lcp_deque.popleft()

            # IF the current window is valid, check its minimum LCP string length
            if distinct_friends == m and left < right:
                if min_lcp_deque:
                    max_common_subpath = max(max_common_subpath, lcp[min_lcp_deque[0]])

            # NOW shrink the window from the left if we have duplicates/unowned values
            # or if we need to advance left to try finding tighter windows
            while distinct_friends == m:
                left_friend = owner[suffix_array[left]]
                if left_friend != -1:
                    if friend_counts[left_friend] == 1:
                        # Dropping this means the window will no longer be valid,
                        # so break out before decrementing distinct_friends
                        break
                    friend_counts[left_friend] -= 1
                left += 1

                # Maintain the deque as left advances
                while min_lcp_deque and min_lcp_deque[0] <= left:
                    min_lcp_deque.popleft()
                if min_lcp_deque:
                    max_common_subpath = max(max_common_subpath, lcp[min_lcp_deque[0]])

        return max_common_subpath

In [30]:
solution = Solution()

In [31]:
n = 5
paths = [[0,1,2,3,4],[2,3,4],[4,0,1,2,3]]

print(solution.longestCommonSubpath(n, paths))

2


In [32]:
n = 3
paths = [[0],[1],[2]]

print(solution.longestCommonSubpath(n, paths))

0


In [33]:
n = 5
paths = [[0,1,2,3,4],[4,3,2,1,0]]

print(solution.longestCommonSubpath(n, paths))

1


In [34]:
n = 5
paths = [[0,1,0,1,0,1,0,1,0],[0,1,3,0,1,4,0,1,0]]

print(solution.longestCommonSubpath(n, paths))

3
